# About dict keys
a Python dictionary can absolutely have a composite key representing a function call, but you cannot use list or dict objects directly as part of that key.

### The Problem: Hashability
Dictionary keys in Python must be hashable (immutable). Because list and dict objects can be modified after they are created, Python prevents them from being used as dictionary keys to avoid corrupting the dictionary's internal lookup tables. If you try, you will get a TypeError: unhashable type.

### The Solution: Immutable Equivalents
To create your composite key, you must convert your mutable arguments into their  immutable equivalents before building the key:

 * name (str): Already immutable. No change needed.

 * positional_arguments (list): Convert this to a tuple. Tuples are immutable lists.

 * keyword_arguments (dict): Convert this to a frozenset of key-value tuples. A frozenset is an immutable set. Because it is a set, it inherently ignores order, which perfectly solves your requirement that keyword arguments in different orders must map to the same key.


In [2]:
from sympy.strategies.core import switch

# Check the MATCH in dict with composite key - e.g.
# ('add', [1, 2], {})
# ('affine', [5], {'scale': 2, 'bias': 1}), or ('affine', [5], {'bias': 1, 'scale': 2})

s_cache:dict = {}
e1 = ('add', [1, 2], {})
e2 = ('mul', [3, 4, 5], {})
e3 = ('pow', [6, 2], {})
e4 = ('add', [1, 2], {}) # same as e1
e5 = ('affine', [5], {'scale': 2, 'bias': 1})
e6 = ('affine', [5], {'bias': 1, 'scale': 2}) # same as e5

# DO NOT WORK!
# s_cache.update({e1: 3})
# s_cache.update(e2, 60)
# s_cache.update(e3, 36)

def immute_key(fn: tuple):
    (first, second, third) = fn
    return first, tuple(second), frozenset(third.items())

s_cache.update({immute_key(e1): 3})
s_cache.update({immute_key(e2): 60})
s_cache.update({immute_key(e3): 36})
s_cache.update({immute_key(e5): 11})
print(s_cache)

# Retrieve!
print(s_cache[immute_key(e4)])
print(s_cache[immute_key(e6)])

{('add', (1, 2), frozenset()): 3, ('mul', (3, 4, 5), frozenset()): 60, ('pow', (6, 2), frozenset()): 36, ('affine', (5,), frozenset({('bias', 1), ('scale', 2)})): 11}
3
11


In [3]:
#

In [4]:
# OrderedDict compatibility, and use for the LRU ops
from collections import OrderedDict

o_cache = OrderedDict(s_cache)
o_cache.move_to_end(immute_key(e1))


In [5]:
# unit test
print(o_cache)
print(o_cache[immute_key(e4)])
print(o_cache[immute_key(e6)])
print(o_cache[immute_key(('add', [1, 2], {}))])
# print(o_cache[immute_key(('add', [2, 1], {}))])  # Key- error!

OrderedDict({('mul', (3, 4, 5), frozenset()): 60, ('pow', (6, 2), frozenset()): 36, ('affine', (5,), frozenset({('bias', 1), ('scale', 2)})): 11, ('add', (1, 2), frozenset()): 3})
3
11
3


In [6]:
# Final impl.
# define functions
def mul(*args, **kwargs):
    prod = 1
    # print(args)
    # print(kwargs)
    for arg in args:
        # print(arg)
        prod *= arg
    return prod

def eval(fn):
    (first, second, third) = fn
    print(f"{first} ==> {second} and {third}")
    if first == 'add':
        return sum(second, **dict(third))
    elif first == 'mul':
        # why *second sometimes, but not other times!!
        return mul(*second, **dict(third))
    elif first == 'pow':
        return second[0] ** second[1]
    elif first == 'affine':
        third = dict(third)
        return second[0] * third['scale'] + third['bias']
    return None

print(eval(immute_key(e1)))
print(eval(immute_key(e2)))
print(eval(immute_key(e3)))
print(eval(immute_key(e5)))


add ==> (1, 2) and frozenset()
3
mul ==> (3, 4, 5) and frozenset()
60
pow ==> (6, 2) and frozenset()
36
affine ==> (5,) and frozenset({('bias', 1), ('scale', 2)})
11


In [11]:
# Final impl.
# LRU cache

def solution(capacity, calls):
    print(f"Capacity {capacity}")
    print(f"Calls {calls}")
    lru_cache = OrderedDict()
    exec_count = 0
    res_list = []
    calls_seq = [immute_key(call) for call in calls]
    for call in calls_seq:
        if call not in lru_cache:
            # drop elements from 0 to (len(lru_cache)-capacity)
            while len(lru_cache) >= capacity:
                lru_cache.popitem(last=False)
            # Evaluate and update cache
            lru_cache[call] = eval(call)
            # count execution
            exec_count += 1
        else:
            ret = lru_cache[call]
            lru_cache.move_to_end(call)
        res_list.append(lru_cache[call])
    return (res_list, exec_count)

In [13]:
# Tests
print(solution(2, [('add', [1, 2], {}), ('add', [1, 2], {})]))
print(solution(2, [('affine', [5], {'scale': 2, 'bias': 1}), ('affine', [5], {'bias': 1, 'scale': 2})]))

Capacity 2
Calls [('add', [1, 2], {}), ('add', [1, 2], {})]
add ==> (1, 2) and frozenset()
([3, 3], 1)
Capacity 2
Calls [('affine', [5], {'scale': 2, 'bias': 1}), ('affine', [5], {'bias': 1, 'scale': 2})]
affine ==> (5,) and frozenset({('bias', 1), ('scale', 2)})
([11, 11], 1)
